In [ ]:
# 1. Reconstruct the original ranking scores
baseline_comparison = ranking_results.copy()

resume_positions = {
    resume_id: i
    for i, resume_id in enumerate(resumes["resume_id"])
}

job_positions = {
    job_id: j
    for j, job_id in enumerate(job_descriptions["job_id"])
}

baseline_comparison["old_tfidf_similarity"] = [
    old_similarity_check[
        resume_positions[resume_id],
        job_positions[job_id]
    ]
    for resume_id, job_id in zip(
        baseline_comparison["resume_id"],
        baseline_comparison["job_id"]
    )
]

baseline_comparison["old_ranking_score"] = 100 * (
    0.40 * baseline_comparison["old_tfidf_similarity"]
    + 0.35 * baseline_comparison["required_skill_coverage"]
    + 0.15 * baseline_comparison["preferred_skill_coverage"]
    + 0.10 * baseline_comparison["experience_relevance"]
)

# 2. Calculate original candidate ranks
old_ranked = baseline_comparison.sort_values(
    ["job_id", "old_ranking_score"],
    ascending=[True, False],
    kind="mergesort"
).copy()

old_ranked["old_model_rank"] = (
    old_ranked.groupby("job_id").cumcount() + 1
)

# 3. Calculate updated candidate ranks
new_ranked = ranking_results.sort_values(
    ["job_id", "ranking_score"],
    ascending=[True, False],
    kind="mergesort"
).copy()

new_ranked["new_model_rank"] = (
    new_ranked.groupby("job_id").cumcount() + 1
)

# 4. Compare old and new rankings
rank_comparison = old_ranked[
    ["resume_id", "job_id", "old_ranking_score", "old_model_rank"]
].merge(
    new_ranked[
        ["resume_id", "job_id", "ranking_score", "new_model_rank"]
    ],
    on=["resume_id", "job_id"],
    how="inner"
)

# 5. Identify rank changes
rank_comparison["rank_changed"] = (
    rank_comparison["old_model_rank"]
    != rank_comparison["new_model_rank"]
)

rank_comparison["top5_changed"] = (
    (rank_comparison["old_model_rank"] <= 5)
    != (rank_comparison["new_model_rank"] <= 5)
)

# 6. Display comparison summary
print("Total comparisons:", len(rank_comparison))

print(
    "Candidates with changed rank:",
    rank_comparison["rank_changed"].sum()
)

print(
    "Candidates with changed top-5 membership:",
    rank_comparison["top5_changed"].sum()
)

print(
    "Average absolute rank movement:",
    round(
        (
            rank_comparison["old_model_rank"]
            - rank_comparison["new_model_rank"]
        ).abs().mean(),
        3
    )
)

# 7. Inspect changed rankings
display(
    rank_comparison[
        rank_comparison["rank_changed"]
    ].sort_values(
        ["job_id", "new_model_rank"]
    ).head(20)
)